# Train YOLO26n on Colab GPU

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Upload to Google Drive folder **`MyDrive/object_detection/`**:
- the dataset zip you want to train (`merged.zip`, `merged_v2.zip`, `merged_v3.zip` or `merged_v4.zip`)
- `train.py`

Training outputs are written straight to `MyDrive/object_detection/runs/` so nothing is lost if Colab disconnects.

In [ ]:
# Choose which dataset to train: 'merged' (v1), 'merged_v2' (v2), 'merged_v3' (v3) or 'merged_v4' (v4: + test split)
DATASET = 'merged_v4'
RUN_NAME = {'merged': 'yolo26n_nav', 'merged_v2': 'yolo26n_nav_v2', 'merged_v3': 'yolo26n_nav_v3', 'merged_v4': 'yolo26n_nav_v4'}[DATASET]

from google.colab import drive
drive.mount('/content/drive')
# Keep this exactly: the mounted Drive path, NOT a https://drive.google.com link
DRIVE = '/content/drive/MyDrive/object_detection'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install the same library versions as the local environment

In [ ]:
!pip install -q ultralytics==8.4.188

## 2. Unzip the dataset to local disk (much faster than reading from Drive)
and point `data.yaml` at the new location.

In [ ]:
import os, re
!mkdir -p /content/work/data && unzip -q -o {DRIVE}/{DATASET}.zip -d /content/work/data
yaml_path = f'/content/work/data/{DATASET}/data.yaml'
text = open(yaml_path).read()
open(yaml_path, 'w').write(re.sub(r'^path: .*$', f'path: /content/work/data/{DATASET}', text, flags=re.M))
print(open(yaml_path).read())
!cp {DRIVE}/train.py /content/work/
# Save runs directly to Drive
os.makedirs(f'{DRIVE}/runs', exist_ok=True)
!ln -sfn {DRIVE}/runs /content/work/runs

## 3. Train
T4 has 15 GB of GPU memory, so batch 32 fits (vs 16 on the 8 GB Mac).

In [ ]:
%cd /content/work
!python train.py --model yolo26n.pt --device 0 --batch 32 --workers 2 --epochs 50 --data {yaml_path} --name {RUN_NAME}

## (Only if Colab disconnected) Resume from the last checkpoint

In [ ]:
# from ultralytics import YOLO
# YOLO(f'{DRIVE}/runs/train/{RUN_NAME}/weights/last.pt').train(resume=True)

## 4. Final validation metrics (val split: used for model selection and thresholds)

In [ ]:
from ultralytics import YOLO
m = YOLO(f'{DRIVE}/runs/train/{RUN_NAME}/weights/best.pt')
metrics = m.val(data=yaml_path, device=0)
print(metrics.box.map50, metrics.box.map)

## 5. Held-out test set (v4 only) — run once, at the very end
The test split was never used for training, model selection or threshold tuning, so this is the unbiased estimate to report.

In [ ]:
if 'test:' in open(yaml_path).read():
    test = m.val(data=yaml_path, split='test', device=0, project=f'{DRIVE}/runs/test', name=RUN_NAME)
    print(test.box.map50, test.box.map)
else:
    print('This dataset has no test split')

## 6. Before stopping or disconnecting: flush Drive
Drive writes are uploaded in the background; this makes sure every checkpoint is saved. Run it whenever you stop training early, too.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

Done — download `MyDrive/object_detection/runs/train/<RUN_NAME>/` and put it at `runs/train/<RUN_NAME>/` in the local repo.
For v4 also download `MyDrive/object_detection/runs/test/yolo26n_nav_v4/` (test-set plots).